<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/11-files-and-formats.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 11 — Files and Formats

Companion to [the chapter](https://www.ai.biz/books/python-primer/files-and-formats/).


In [ ]:
import pandas as pd, numpy as np, io, tempfile, os
print('pandas', pd.__version__)


## 1. The identifier disaster

Pandas sees digits and makes an integer. The leading zeros are gone forever.


In [ ]:
csv = 'customer_id,postcode,revenue\n0012345,01234,100.5\n0067890,00987,200.0\n'

bad = pd.read_csv(io.StringIO(csv))
good = pd.read_csv(io.StringIO(csv), dtype={'customer_id':'string','postcode':'string'})

print('guessed :', bad.customer_id.tolist(), '|', bad.customer_id.dtype)
print('explicit:', good.customer_id.tolist(), '|', good.customer_id.dtype)
print()
print('The first will never join to anything again.')


**The test:** would you ever add two of them together? If not, it is text.


## 2. Look at the raw bytes before parsing

`repr` shows the invisible characters that break parsing.


In [ ]:
messy = 'name,value\r\nDelhi ,10\r\n Tokyo,20\r\n'
for line in io.StringIO(messy):
    print(repr(line))
print()
print('Note the \\r\\n line endings and the trailing/leading spaces.')


## 3. na_values and thousands separators


In [ ]:
csv = 'city,revenue,notes\nDelhi,"1,200",ok\nSydney,N/A,-\nTokyo,"3,400",\n'

plain = pd.read_csv(io.StringIO(csv))
print('revenue dtype without args:', plain.revenue.dtype, '<- object, useless')

clean = pd.read_csv(io.StringIO(csv), thousands=',',
                    na_values=['', 'N/A', '-', 'null', 'NA'])
print('revenue dtype with args   :', clean.revenue.dtype)
print(clean)


## 4. The fifteen-minute check

Run this before forming any opinion about a dataset.


In [ ]:
rng = np.random.default_rng(0)
df = pd.DataFrame({
    'country': rng.choice(['UK','U.K.','United Kingdom','england','Uk '], 500),
    'revenue': rng.lognormal(5, 1, 500).round(2),
    'region': rng.choice(['Asia','Europe',None], 500, p=[.5,.4,.1]),
})

print('shape:', df.shape)
print('exact duplicate rows:', df.duplicated().sum())
print()
print('missing per column:'); print(df.isna().mean().round(3))
print()
for c in df.select_dtypes(['object']):
    print(f'{c}: {df[c].nunique()} unique')
    print(df[c].value_counts().head())
    print()


Five spellings of one country. A model would treat them as five unrelated categories.


## 5. Parquet preserves types; CSV does not


In [ ]:
d = pd.DataFrame({
    'id': pd.Series(['001','002'], dtype='string'),
    'when': pd.to_datetime(['2026-01-01','2026-02-01']),
    'region': pd.Series(['a','b'], dtype='category'),
})
print('original dtypes:'); print(d.dtypes)

tmp = tempfile.gettempdir()
d.to_csv(os.path.join(tmp,'t.csv'), index=False)
print()
print('after CSV round trip:'); print(pd.read_csv(os.path.join(tmp,'t.csv')).dtypes)

try:
    d.to_parquet(os.path.join(tmp,'t.parquet'))
    print(); print('after Parquet round trip:')
    print(pd.read_parquet(os.path.join(tmp,'t.parquet')).dtypes)
except ImportError:
    print(); print('(install pyarrow to run the Parquet half)')


## 6. Chunked reading for files larger than memory


In [ ]:
big = pd.DataFrame({'region': rng.choice(list('ABCD'), 50_000),
                    'revenue': rng.random(50_000) * 100})
path = os.path.join(tmp, 'big.csv'); big.to_csv(path, index=False)

totals = []
for chunk in pd.read_csv(path, chunksize=10_000):
    totals.append(chunk.groupby('region').revenue.sum())

chunked = pd.concat(totals).groupby(level=0).sum()
whole = big.groupby('region').revenue.sum()
print('identical:', np.allclose(chunked.sort_index(), whole.sort_index()))


Works for sums, counts and group totals. **Does not work for a median**, which needs all the data at once.


## Try it yourself

1. Read the messy CSV above with `encoding='latin-1'` and see that it never raises, even on bad input.
2. Time `read_csv` against `read_parquet` on the 50,000-row file.
3. Normalise the five country spellings and recount.
